# Ordinal Encoding and Label Encoding

Categorical data contains values represented by labels or categories rather than directly usable numeric measurements. Many machine-learning algorithms require numerical input, so categorical features and target labels often need to be encoded.

This notebook demonstrates two techniques using a customer dataset:

- **Ordinal Encoding** converts ordered categorical input features into numbers while preserving their specified order.
- **Label Encoding** converts target class labels into integer values.

The example uses `review` and `education` as input features and `purchased` as the target.

In [1]:
import pandas as pd , numpy as np 

In [2]:
df = pd.read_csv('Datasets/customer.csv')

In [3]:
df.head()

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


After removing the `age` and `gender` columns, the dataset contains:

- **Ordinal features:** `review` (`Poor`, `Average`, `Good`) and `education` (`School`, `UG`, `PG`). Their categories have a meaningful order.
- **Nominal target:** `purchased` (`No`, `Yes`). These are class labels, not ordered levels.

Ordinal encoding is appropriate for the two input features because their category order is known. The target is encoded separately with `LabelEncoder`.

In [4]:
df = df.iloc[:, 2:]
df.head()

,review,education,purchased
0,Average,School,No
1,Poor,UG,No
2,Good,PG,No
3,Good,PG,No
4,Average,UG,No


### Ordinal Encoder 

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 0:2], df.iloc[:,-1], test_size=0.2)

In [6]:
X_train

,review,education
34,Average,School
44,Average,UG
12,Poor,School
8,Average,UG
32,Average,UG
23,Good,School
26,Poor,PG
5,Average,School
15,Poor,UG
28,Poor,School


In [7]:
from sklearn.preprocessing import OrdinalEncoder

#### Define the category order

`categories` specifies the order for each input column, in the same order as the columns in `X_train`:

| Feature | Specified order | Encoded values |
|---|---|---|
| `review` | Poor → Average → Good | 0 → 1 → 2 |
| `education` | School → UG → PG | 0 → 1 → 2 |

The numeric values represent rank/order, not measured distances. For example, the difference between `Poor` and `Average` is not guaranteed to be equal in meaning to the difference between `Average` and `Good`.

In [8]:
# here we tell the order, in asceding order, means poor ko sabse km value and Good ko sabse se jayda. 
oe = OrdinalEncoder(categories=[['Poor', 'Average', 'Good'],['School', 'UG', 'PG']])

In [9]:
oe.fit(X_train)

OrdinalEncoder(categories=[['Poor', 'Average', 'Good'], ['School', 'UG', 'PG']])

In [10]:
X_train = oe.transform(X_train)
X_test = oe.transform(X_test)

In [11]:
X_train

array([[1., 0.],
       [1., 1.],
       [0., 0.],
       [1., 1.],
       [1., 1.],
       [2., 0.],
       [0., 2.],
       [1., 0.],
       [0., 1.],
       [0., 0.],
       [1., 0.],
       [2., 0.],
       [1., 0.],
       [2., 2.],
       [2., 0.],
       [0., 2.],
       [2., 0.],
       [0., 1.],
       [2., 2.],
       [2., 2.],
       [2., 2.],
       [2., 1.],
       [2., 2.],
       [0., 0.],
       [0., 1.],
       [2., 1.],
       [2., 2.],
       [0., 2.],
       [1., 0.],
       [2., 0.],
       [2., 1.],
       [2., 1.],
       [0., 2.],
       [1., 2.],
       [1., 2.],
       [0., 0.],
       [0., 2.],
       [0., 2.],
       [2., 0.],
       [0., 0.]])

### Label Encoding

`LabelEncoder` converts the target labels into integer class identifiers. In this example, `purchased` contains `No` and `Yes`.

**Important distinction :** use `OrdinalEncoder` for categorical input features (`X`) and `LabelEncoder` for a one-dimensional target (`y`). `LabelEncoder` does not preserve a meaningful ranking; it simply assigns integer identifiers to classes.

In [12]:
from sklearn.preprocessing import LabelEncoder

In [13]:
le = LabelEncoder()

In [14]:
le.fit(y_train)

LabelEncoder()

For this dataset, the classes are ordered alphabetically, so the mapping is:

- `No` → `0`
- `Yes` → `1`

These numbers are class identifiers; they do not indicate that `Yes` is twice `No`.

In [15]:
le.classes_

array(['No', 'Yes'], dtype=object)

In [16]:
y_train = le.transform(y_train)
y_test = le.transform(y_test)

The output now contains integer class labels. These encoded values can be supplied to a classification model. When interpreting predictions, use `le.inverse_transform()` to convert numeric class identifiers back to their original labels.

In [17]:
y_train

array([0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1])

### Advantages and limitations

**Ordinal Encoding**

- Preserves the meaningful order of categories.
- Produces numeric values that many ML algorithms can process.
- The assigned numbers may imply an ordered relationship to a model.
- An incorrect or arbitrary category order can mislead the model.
- It is not suitable for nominal categories with no natural order, because numeric codes can introduce a false ranking.

**Label Encoding**

- Simple way to convert target class names into integers.
- Works well for classification targets supported by scikit-learn.
- The integer values are identifiers, not quantities or ranks.
- Do not use it for nominal input features when the numeric codes could make a model assume an artificial order.

### Key takeaways

1. Identify whether a categorical variable is ordinal or nominal before encoding it.
2. Use `OrdinalEncoder` for ordered input features and explicitly define the category order when it is known.
3. Fit encoders on training data, then use the fitted encoders to transform both training and test data.
4. Use `LabelEncoder` for target labels in this example.
5. Keep the fitted encoders so that future or production data is transformed using exactly the same mappings.